In [ ]:
pip install numpy scikit-learn

In [3]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



In [21]:
class Perceptron:
  def __init__(self, learning_rate=0.01, n_epochs=100, random_state=42):
    self.learning_rate=learning_rate
    self.n_epochs=n_epochs
    self.random_state=random_state

    self.weights=None
    self.bias=None
    self.errors_per_epoch=[]

  def fit(self, X, y): #entraine le perceptron sur les données X et labels Y
    n_samples, n_features=X.shape
    self.weights = np.zeros(n_features)
    self.bias=0.0
    self.errors_per_epoch=[] #chaque entrainement commence avec des paramètres nuls

    rng= np.random.default_rng(self.random_state) #générateur utilisé pour mélanger les exemples à chaque epoch
    for epoch in range(self.n_epochs):
      errors=0
      indices=rng.permutation(n_samples) #utiliser le générateur dans le modèle
      for i in indices:
        x_i=X[i]
        y_i=y[i]

        score=np.dot(x_i, self.weights)+self.bias #calcul de score
        y_pred=1 if score >= 0 else 0 #transformation du score en une prédiction binaire
        error=y_i - y_pred
        update=self.learning_rate*error #calcul de l'erreur et update pour la correction
        self.weights += update*x_i
        self.bias += update #modifie les paramètres si nécessaire

        if error != 0:
          errors +=1 #si les erreurs existent alors on incrémente 1 de plus

      self.errors_per_epoch.append(errors) #et on append dans la liste par epoch

      if errors ==0:
        break #si tous les exemples ont été correctement classés pendant cet epoch, aucune correction nécessaire

  def predict(self, X): #prédiction de la classe de chaque observation de X
    scores=np.dot(X, self.weights)+self.bias
    return (scores>=0).astype(int)

## Charger et explorer le jeu de données

In [5]:
dataset = load_breast_cancer()

In [6]:
X=dataset.data
y=dataset.target

print("Shape de X: ", X.shape)
print("Shape de Y: ", y.shape)

Shape de X:  (569, 30)
Shape de Y:  (569,)


In [18]:
print("Classes: ", dataset.target_names)
print("Premières caractéristiques: ", dataset.feature_names[:5])

Classes:  ['malignant' 'benign']
Premières caractéristiques:  ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness']


## Séparer les données d'entraînement et de test

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Standardisation des caractéristiques

In [11]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)


In [12]:
X_test_scaled=scaler.transform(X_test)

## Entrainer le perceptron

In [22]:
model = Perceptron(learning_rate=0.01, n_epochs=100, random_state=42)
model.fit(X_train_scaled, y_train)

## Evaluation du modèle sur données de test

In [39]:
y_pred = model.predict(X_test_scaled)

print("\nNombre d'epochs effectuées :", len(model.errors_per_epoch))
print("Erreurs pendant les premières epochs :", model.errors_per_epoch[:100])
print("Erreurs pendant la dernière epochs :", model.errors_per_epoch[-1])

print("\nAccuracy :", accuracy_score(y_test, y_pred))

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))

print("\nRapport de classification :")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=dataset.target_names,
        zero_division=0,
    )
)


Nombre d'epochs effectuées : 100
Erreurs pendant les premières epochs : [24, 14, 19, 13, 13, 9, 7, 12, 9, 8, 17, 10, 11, 12, 5, 13, 16, 11, 8, 9, 9, 11, 8, 11, 7, 12, 12, 7, 14, 11, 8, 9, 9, 10, 6, 7, 13, 15, 12, 6, 9, 11, 11, 9, 12, 8, 8, 9, 8, 12, 9, 10, 13, 11, 8, 9, 6, 10, 10, 11, 9, 11, 9, 8, 6, 13, 8, 12, 8, 4, 11, 6, 7, 12, 12, 6, 11, 11, 12, 8, 14, 10, 6, 10, 6, 13, 9, 11, 8, 10, 9, 9, 11, 9, 9, 4, 15, 7, 13, 8]
Erreurs pendant la dernière epochs : 8

Accuracy : 0.9824561403508771

Matrice de confusion :
[[40  2]
 [ 0 72]]

Rapport de classification :
              precision    recall  f1-score   support

   malignant       1.00      0.95      0.98        42
      benign       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

